# MMF Hierarchical Reconciliation
Auto-generated by mmf-agent.

In [ ]:
%pip install "mmf_sa[hierarchical] @ git+https://github.com/databricks-industry-solutions/many-model-forecasting.git@main" --quiet

In [ ]:
%restart_python

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
date_col = "{date_col}"
target = "{target}"
reconciliation_method = "{reconciliation_method}"  # BottomUp | TopDown | MiddleOut | MinTrace | ERM

In [ ]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)

from mmf_sa import run_reconciliation

# MinTrace and ERM require fitted_output from run_forecast().
# BottomUp and TopDown work without it.
_fitted_output = catalog + "." + schema + "." + use_case + "_fitted_output"
_uses_fitted = reconciliation_method in ("MinTrace", "ERM")

run_reconciliation(
    spark=spark,
    best_models_table=catalog + "." + schema + "." + use_case + "_best_models",
    hierarchy_s_table=catalog + "." + schema + "." + use_case + "_hierarchy_S",
    hierarchy_tags_table=catalog + "." + schema + "." + use_case + "_hierarchy_tags",
    fitted_output=_fitted_output if _uses_fitted else None,
    reconciliation_output=catalog + "." + schema + "." + use_case + "_reconciliation_output",
    date_col=date_col,
    target=target,
    method=reconciliation_method,
)

In [ ]:
# Quick coherence spot-check — sample one date and verify sum of children matches parent
result = spark.table(catalog + "." + schema + "." + use_case + "_reconciliation_output")
display(result.limit(20))
print(f"Reconciliation output: {result.count()} rows")
print(f"Hierarchy levels: {[r[0] for r in result.select('hierarchy_level').distinct().collect()]}")